In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
static_file = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_01_withBias_1h_stat_low_high.txt"

dynamic_file = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_04_withBias_1h_dyn_low_high.txt"

df_static = pd.read_csv(static_file)
df_dynamic = pd.read_csv(dynamic_file)

print("Static :", df_static.shape)
print("Dynamic:", df_dynamic.shape)

Static : (180000, 18)
Dynamic: (360000, 18)


In [3]:
for axis in ["x", "y", "z"]:
    df_static[f"error_{axis}"] = (
        df_static[f"IMU_low_a{axis}"] -
        df_static[f"IMU_high_a{axis}"]
    )

    df_dynamic[f"error_{axis}"] = (
        df_dynamic[f"IMU_low_a{axis}"] -
        df_dynamic[f"IMU_high_a{axis}"]
    )

In [4]:
feature_cols = [
    "IMU_low_ax",
    "IMU_low_ay",
    "IMU_low_az"
]

target_cols = [
    "error_x",
    "error_y",
    "error_z"
]

X_train = df_static[feature_cols].values
Y_train = df_static[target_cols].values

X_test = df_dynamic[feature_cols].values
Y_test = df_dynamic[target_cols].values

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test :", X_test.shape)
print("Y_test :", Y_test.shape)

X_train: (180000, 3)
Y_train: (180000, 3)
X_test : (360000, 3)
Y_test : (360000, 3)


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Ridge(alpha=1.0)

model.fit(X_train_scaled, Y_train)

Y_pred = model.predict(X_test_scaled)

In [6]:
for i, axis in enumerate(["X", "Y", "Z"]):

    baseline_pred = np.full(
        len(Y_test),
        Y_train[:, i].mean()
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            baseline_pred
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            Y_pred[:, i]
        )
    )

    ml_mae = mean_absolute_error(
        Y_test[:, i],
        Y_pred[:, i]
    )

    improvement = (
        1 - ml_rmse / baseline_rmse
    ) * 100

    print(f"\n{axis}-axis")
    print(f"Baseline RMSE : {baseline_rmse:.6f}")
    print(f"ML RMSE       : {ml_rmse:.6f}")
    print(f"ML MAE        : {ml_mae:.6f}")
    print(f"Improvement   : {improvement:.2f}%")


X-axis
Baseline RMSE : 0.217475
ML RMSE       : 5.635994
ML MAE        : 4.873881
Improvement   : -2491.56%

Y-axis
Baseline RMSE : 0.216407
ML RMSE       : 5.614529
ML MAE        : 4.823177
Improvement   : -2494.44%

Z-axis
Baseline RMSE : 0.186020
ML RMSE       : 11.433798
ML MAE        : 9.889022
Improvement   : -6046.56%


In [7]:
print("STATIC acceleration ranges")
print(df_static[feature_cols].describe().loc[["mean", "std", "min", "max"]])

print("\nDYNAMIC acceleration ranges")
print(df_dynamic[feature_cols].describe().loc[["mean", "std", "min", "max"]])

STATIC acceleration ranges
      IMU_low_ax  IMU_low_ay  IMU_low_az
mean   -1.867050    2.227379   10.696854
std     0.053578    0.051571    0.083315
min    -2.057900    2.084000   10.503000
max    -1.716300    2.381500   10.948000

DYNAMIC acceleration ranges
      IMU_low_ax  IMU_low_ay  IMU_low_az
mean   -1.960964    2.085293    0.886643
std     5.634226    5.613797    5.735836
min   -11.944000   -7.839200   -9.025800
max     7.885100   12.174000   10.892000


In [8]:
print("\nStatic target ranges")
print(df_static[target_cols].describe().loc[["mean", "std", "min", "max"]])

print("\nDynamic target ranges")
print(df_dynamic[target_cols].describe().loc[["mean", "std", "min", "max"]])


Static target ranges
       error_x   error_y   error_z
mean -1.867105  2.227332  0.886849
std   0.053651  0.051642  0.083363
min  -2.058673  2.083275  0.692500
max  -1.715456  2.385355  1.137100

Dynamic target ranges
       error_x   error_y   error_z
mean -2.005781  2.106845  0.965404
std   0.167525  0.179763  0.168619
min  -2.438300  1.661800  0.466920
max  -1.584390  2.649480  1.374440


In [9]:
# Use the same instantaneous features
feature_cols = [
    "IMU_low_ax",
    "IMU_low_ay",
    "IMU_low_az"
]

target_cols = [
    "error_x",
    "error_y",
    "error_z"
]

# Static data
X_static = df_static[feature_cols].values
Y_static = df_static[target_cols].values

# Dynamic data
X_dynamic = df_dynamic[feature_cols].values
Y_dynamic = df_dynamic[target_cols].values

print("Static :", X_static.shape, Y_static.shape)
print("Dynamic:", X_dynamic.shape, Y_dynamic.shape)

Static : (180000, 3) (180000, 3)
Dynamic: (360000, 3) (360000, 3)


In [10]:
static_split = int(0.8 * len(X_static))
dynamic_split = int(0.8 * len(X_dynamic))

X_static_train = X_static[:static_split]
Y_static_train = Y_static[:static_split]

X_dynamic_train = X_dynamic[:dynamic_split]
Y_dynamic_train = Y_dynamic[:dynamic_split]

X_dynamic_test = X_dynamic[dynamic_split:]
Y_dynamic_test = Y_dynamic[dynamic_split:]

# Combine training conditions
X_train_mixed = np.vstack([
    X_static_train,
    X_dynamic_train
])

Y_train_mixed = np.vstack([
    Y_static_train,
    Y_dynamic_train
])

print("Mixed training:", X_train_mixed.shape)
print("Dynamic test  :", X_dynamic_test.shape)

Mixed training: (432000, 3)
Dynamic test  : (72000, 3)


In [11]:
scaler_mixed = StandardScaler()

X_train_mixed_scaled = scaler_mixed.fit_transform(X_train_mixed)
X_dynamic_test_scaled = scaler_mixed.transform(X_dynamic_test)

model_mixed = Ridge(alpha=1.0)

model_mixed.fit(
    X_train_mixed_scaled,
    Y_train_mixed
)

Y_pred_mixed = model_mixed.predict(
    X_dynamic_test_scaled
)

In [12]:
for i, axis in enumerate(["X", "Y", "Z"]):

    baseline_pred = np.full(
        len(Y_dynamic_test),
        Y_train_mixed[:, i].mean()
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_dynamic_test[:, i],
            baseline_pred
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            Y_dynamic_test[:, i],
            Y_pred_mixed[:, i]
        )
    )

    ml_mae = mean_absolute_error(
        Y_dynamic_test[:, i],
        Y_pred_mixed[:, i]
    )

    improvement = (
        1 - ml_rmse / baseline_rmse
    ) * 100

    print(f"\n{axis}-axis")
    print(f"Baseline RMSE : {baseline_rmse:.6f}")
    print(f"ML RMSE       : {ml_rmse:.6f}")
    print(f"ML MAE        : {ml_mae:.6f}")
    print(f"Improvement   : {improvement:.2f}%")


X-axis
Baseline RMSE : 0.166507
ML RMSE       : 0.101666
ML MAE        : 0.092628
Improvement   : 38.94%

Y-axis
Baseline RMSE : 0.180819
ML RMSE       : 0.090223
ML MAE        : 0.070712
Improvement   : 50.10%

Z-axis
Baseline RMSE : 0.181121
ML RMSE       : 0.103443
ML MAE        : 0.085886
Improvement   : 42.89%


In [13]:
from sklearn.ensemble import RandomForestRegressor

In [14]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_mixed,
    Y_train_mixed
)

Y_pred_rf = rf_model.predict(
    X_dynamic_test
)

In [ ]:
for i, axis in enumerate(["X", "Y", "Z"]):

    # Baseline = mean error of mixed training data
    baseline_value = Y_train_mixed[:, i].mean()

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_dynamic_test[:, i],
            np.full(len(Y_dynamic_test), baseline_value)
        )
    )

    rf_rmse = np.sqrt(
        mean_squared_error(
            Y_dynamic_test[:, i],
            Y_pred_rf[:, i]
        )
    )

    rf_mae = mean_absolute_error(
        Y_dynamic_test[:, i],
        Y_pred_rf[:, i]
    )

    improvement = (
        1 - rf_rmse / baseline_rmse
    ) * 100

    print(f"\n{axis}-axis")
    print(f"Baseline RMSE : {baseline_rmse:.6f}")
    print(f"RF RMSE       : {rf_rmse:.6f}")
    print(f"RF MAE        : {rf_mae:.6f}")
    print(f"Improvement   : {improvement:.2f}%")

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed